# Phase 1 (Kaggle): pilot comparison — Qwen3.5-4B vs Gemma-4-E4B

Identical pilot recipe for both models (2,002 train examples, 26/intent, same seed, same LoRA settings,
same prompt rendering with thinking OFF). Only the model changes.

**Kaggle setup (do this before running):**
1. Create a Dataset from `kaggle_upload/banking-finetuning-code.zip` and **add it to this notebook** (Add Input).
2. Notebook settings: **Accelerator = GPU T4 x2**, **Internet = On** (needs a phone-verified account).
3. Run cells top to bottom.

With 2 GPUs, the two models run **at the same time**, one per GPU.

In [ ]:
import os, glob, shutil, subprocess, sys
# Kaggle nests uploads differently across versions (and may leave the zip unextracted), so search recursively.
def find_src():
    hits = glob.glob('/kaggle/input/**/src/schema.py', recursive=True)
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        shutil.unpack_archive(z, '/kaggle/working/_code')
        hits = glob.glob('/kaggle/working/_code/**/src/schema.py', recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(hits[0]))
    print('Contents of /kaggle/input:'); print(subprocess.run('find /kaggle/input -maxdepth 4 | head -40', shell=True, capture_output=True, text=True).stdout)
    raise FileNotFoundError('code dataset not found: add banking-finetuning-code via Add Input')

SRC_ROOT = find_src()
PROJ = '/kaggle/working/Banking_FInetuning'
# Always refresh code/configs from the dataset (keeps data/ and outputs/ already in PROJ).
shutil.copytree(SRC_ROOT, PROJ, ignore=shutil.ignore_patterns('__pycache__'), dirs_exist_ok=True)
os.chdir(PROJ)
os.makedirs('logs', exist_ok=True)
os.environ['HF_HOME'] = '/kaggle/tmp/hf'      # model downloads: keep out of /kaggle/working (20 GB cap)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'   # less fragmentation / fewer OOMs
print('project at', PROJ, '| code from', SRC_ROOT)
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

In [ ]:
# Freeze Kaggle's CUDA build of torch so pip can't silently replace it with something else.
!pip list --format=freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/constraints.txt
!cat /tmp/constraints.txt
!pip install -q -U -c /tmp/constraints.txt transformers peft accelerate bitsandbytes mlflow scikit-learn pyyaml pandas datasets

In [ ]:
# GPU check — must pass before anything else. If it fails, read the printed lines.
import subprocess, torch, transformers, peft, bitsandbytes
print('torch', torch.__version__, '| built for CUDA:', torch.version.cuda, '| cuda available:', torch.cuda.is_available(),
      '| GPUs:', torch.cuda.device_count())
print('transformers', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__)
print(subprocess.run('nvidia-smi -L; pip list 2>/dev/null | grep -iE "^(torch|nvidia-cuda-runtime)"', shell=True, capture_output=True, text=True).stdout)
assert torch.version.cuda is not None, 'CPU-only torch is installed (see pip list above)'
assert torch.cuda.is_available(), 'No GPU visible: set Accelerator to GPU T4 x2 in the notebook settings, then restart the session'

### Optional: fast kernels for Qwen3.5
Qwen3.5 uses a "gated delta" attention layer. Without the `flash-linear-attention` package it falls back to a very
slow pure-PyTorch version (the smoke-test log warns about this). Installing it can cut Qwen's time a lot.
If this cell errors, just skip it: results are identical, only slower. (`causal_conv1d` needs compiling; skip it.)

In [ ]:
!pip install -q -c /tmp/constraints.txt flash-linear-attention

In [ ]:
# Helpers
import json, threading, time
MODELS = ['qwen3_5_4b', 'gemma4_e4b']
cfg = lambda m: f'configs/pilot/{m}.yaml'

def run(module, *args, gpu=None, log=None):
    cmd = [sys.executable, '-m', module, *map(str, args)]
    env = dict(os.environ)
    if gpu is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    if log is None:   # stream into the notebook
        print('$', ' '.join(cmd))
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in p.stdout:
            print(line, end='')
        rc = p.wait()
    else:
        log.write('$ ' + ' '.join(cmd) + '\n'); log.flush()
        rc = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
    return rc

def run_models_parallel(steps_for, tag, models=None):
    """steps_for(model) -> list of (module, args). One model per GPU if 2 GPUs, else one after another."""
    models = models or MODELS
    ngpu = torch.cuda.device_count()
    results = {}
    def worker(m, gpu):
        with open(f'logs/{tag}_{m}.log', 'w') as log:
            for module, args in steps_for(m):
                rc = run(module, *args, gpu=gpu, log=log)
                if rc:
                    results[m] = f'FAILED in {module} (exit {rc})'; return
        results[m] = 'ok'
    def clean(m):   # log lines without tqdm progress-bar noise
        f = f'logs/{tag}_{m}.log'
        return [l for l in open(f).read().splitlines() if 'it/s' not in l and 's/it' not in l] if os.path.exists(f) else []
    def show():
        for m in models:
            print(f'[{m}]', ' | '.join(l[:150] for l in clean(m)[-2:]))
    if ngpu >= 2:
        ts = [threading.Thread(target=worker, args=(m, i)) for i, m in enumerate(models)]
        [t.start() for t in ts]
        while any(t.is_alive() for t in ts):
            time.sleep(60); show(); print('---')
        [t.join() for t in ts]
    else:
        for m in models:
            worker(m, 0); show()
    print(results)
    for m, r in results.items():
        if r != 'ok':
            print(f'\n===== {m}: {r} - last log lines =====')
            print('\n'.join(clean(m)[-45:]))
    return results

## 1. Data
Downloads Banking77 (needs Internet On) and builds the pilot/test splits.

In [ ]:
if not os.path.exists('data/stats.json'):
    assert run('src.data.prepare') == 0
for s in json.load(open('data/stats.json')):
    print(s['split'].ljust(12), 'n =', s['n'], '| per-intent', s['min_per_intent'], '-', s['max_per_intent'])

## 2. Sanity: what the models actually see
Check that the prompt ends with the "thinking off" marker and the answer ends with the end-of-turn token.

In [ ]:
from transformers import AutoTokenizer
from src.config import load_cfg
from src.formatting import build_renderer
row = json.loads(open('data/pilot_train.jsonl').readline())
for m in MODELS:
    c = load_cfg(cfg(m)); tok = AutoTokenizer.from_pretrained(c['base_model']); r = build_renderer(tok)
    print('=====', c['base_model'], '| stop ids:', r.end_ids)
    print('PROMPT    :', repr(r.prompt(row['prompt'])[-160:]))
    print('COMPLETION:', repr(r.completion(row['completion'][0]['content'])))

## 3. Smoke test
Trains on 64 rows and generates for 32. Catches loading, LoRA-targeting, fp16/NaN and forward-pass errors
before the long runs. **Watch the loss in the log: it must be a finite number (not `nan`).**
T4 has no bf16, so both models run in fp16 here, which is the main thing this checks.
If anything fails, paste the error back to Claude.

In [ ]:
def smoke_steps(m):
    return [('src.train.qlora', ['--config', cfg(m), '--limit', 64]),
            ('src.eval.hf_eval', ['--config', cfg(m), '--split', 'data/test_small.jsonl', '--limit', 32,
                                  '--batch-size', 16, '--adapter', f'outputs/pilot/{m}/adapter', '--tag', 'smoke'])]
smoke = run_models_parallel(smoke_steps, 'smoke')

In [ ]:
# Inspect the smoke logs if anything above failed or looks odd
for m in MODELS:
    print('=====', m); print(''.join(open(f'logs/smoke_{m}.log').readlines()[-40:]))

## 4. Full pilot (both models)
Per model: zero-shot baseline (label list in prompt, 770 examples) → QLoRA on the pilot set → full 3,080-example test.
Progress is printed every minute.

In [ ]:
def pilot_steps(m):
    return [('src.eval.hf_eval', ['--config', cfg(m), '--split', 'data/test_small.jsonl', '--zero-shot',
                                  '--batch-size', 16, '--tag', 'zeroshot_small']),
            ('src.train.qlora', ['--config', cfg(m)]),
            ('src.eval.hf_eval', ['--config', cfg(m), '--split', 'data/test.jsonl', '--batch-size', 16,
                                  '--adapter', f'outputs/pilot/{m}/adapter', '--tag', 'finetuned_test'])]
pilot = run_models_parallel(pilot_steps, 'pilot')

## 5. Decision
Rule from PLAN.md: higher macro-F1 wins (needs ≥98% valid JSON); within the noise band (paired bootstrap CI
includes 0, or diff < 0.01) it is a tie → faster model, and re-run with a second seed.

In [ ]:
assert run('src.eval.compare', '--models', *MODELS) == 0

## 6. Error analysis
Which intents does each model confuse most?

In [ ]:
import collections
for m in MODELS:
    preds = [json.loads(l) for l in open(f'outputs/pilot/{m}/finetuned_test_preds.jsonl')]
    wrong = collections.Counter((p['gold'], p['pred']) for p in preds if p['gold'] != p['pred'])
    print('=====', m, '| invalid outputs:', sum(p['pred'] == '__invalid__' for p in preds))
    for (g, p), n in wrong.most_common(8):
        print(f'  {n:>3}  {g}  ->  {p}')

## 7. Save results
Everything small and important is in `outputs/pilot/*` (adapters, metrics, predictions) and `results/pilot/comparison.md`.
Use **Save Version → Save & Run All (Commit)** or download these files before the session ends; Kaggle does not keep
`/kaggle/working` after an interactive session closes.

In [ ]:
!cd /kaggle/working && zip -qr pilot_results.zip Banking_FInetuning/outputs/pilot Banking_FInetuning/results Banking_FInetuning/logs Banking_FInetuning/data/stats.json -x '*/trainer/*' && ls -lh pilot_results.zip